## **Conexión a BlobServiceClient**

In [2]:
from pathlib import Path
from azure.storage.blob import BlobServiceClient

# 1. Definimos la ruta usando pathlib
ruta_llave = Path("key.txt")

# 2. Validación preventiva: ¿Existe el archivo realmente?
if not ruta_llave.exists():
    print(f"ERROR: No se encontró el archivo de credenciales en la ruta: '{ruta_llave.resolve()}'")
else:
    try:
        # 3. Si el archivo existe, leemos la cadena y limpiamos espacios
        connect_str = ruta_llave.read_text(encoding="utf-8").strip()

        # 4. Intentamos crear el cliente del servicio
        blob_service_client = BlobServiceClient.from_connection_string(connect_str)
        
        print("¡Éxito! Conectado a Azure Storage.")
        
    except ValueError as ve:
        # Error específico: La cadena de conexión está mal escrita o incompleta
        print(f"Error de formato en la cadena de conexión: {ve}")
    except Exception as e:
        # Error general de conexión
        print(f"Error inesperado al conectar con Azure: {e}")

¡Éxito! Conectado a Azure Storage.


## **Creación del Contenedor**

In [3]:
container_name = "zona-bronze" # ojo sólo se permiten letras minúsculas, números y guiones medios

try:
    # Crear el contenedor
    container_client = blob_service_client.create_container(container_name)
    print(f"Contenedor '{container_name}' creado.")
except Exception as e:
    print(f"El contenedor ya existe o hubo un error: {e}")

Contenedor 'zona-bronze' creado.


## **Subir Datos (Upload Blob)**

In [4]:
# Obtener referencia al contenedor si es que ya existe
container_client = blob_service_client.get_container_client("zona-bronze")

local_file = "../data/sales.csv"
adls_path = "raw/ventas/2026/07/sales.csv"

with open(local_file, "rb") as data: # rb: modo lectura binaria
    container_client.upload_blob(name = adls_path, data = data, overwrite = True)

print("Archivo subido a la ruta jerárquica: {adls_path}.")

Archivo subido a la ruta jerárquica: {adls_path}.


## **Listar archivos**

In [5]:
print("Listando archivos en zona-bronze:")

# Filtrar por carpeta con nombre name_starts_with
blob_list = container_client.list_blobs(name_starts_with = "raw/ventas/")

for blob in blob_list:
    print(f"Nombre: {blob.name}, Tamaño: {blob.size} bytes.")

Listando archivos en zona-bronze:
Nombre: raw/ventas/2026, Tamaño: 0 bytes.
Nombre: raw/ventas/2026/07, Tamaño: 0 bytes.
Nombre: raw/ventas/2026/07/sales.csv, Tamaño: 2129689 bytes.


## **Descargar Datos**

In [6]:
blob_client = container_client.get_blob_client("raw/ventas/2026/07/sales.csv")

with open("../data/datos_descargados.csv", "wb") as my_blob:
    download_stream = blob_client.download_blob()
    my_blob.write(download_stream.readall())

print("Archivo descargado.")

Archivo descargado.


## **Desafío Práctico Azure**

In [7]:
import os
from datetime import datetime

# Crear carpeta de prueba
os.makedirs("../datos_locales", exist_ok=True)

# Lista de fechas falsas para distribuir los archivos
fechas_falsas = [
    (2023, 10, 15), (2023, 11, 2),
    (2024, 1, 5), (2024, 1, 6),
    (2026, 7, 8) 
]

for i in range(10):
    ruta_archivo = f"../datos_locales/ticket_{i}.csv"
    
    # Escribir un CSV básico
    with open(ruta_archivo, "w", encoding="utf-8") as f:
        f.write("id_venta,monto\n1,100")

    # Truco de sistema: Sobrescribir la fecha de modificación del archivo
    año, mes, dia = fechas_falsas[i % len(fechas_falsas)]
    fecha_dt = datetime(año, mes, dia, 12, 0, 0)
    timestamp = fecha_dt.timestamp()
    
    # Cambia los metadatos locales (Access time, Modified time)
    os.utime(ruta_archivo, (timestamp, timestamp))



In [8]:
import time
import logging

# Configuración de Logging (Reemplaza los print básicos)
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

"""
silenciar consola
# Cambiar INFO por WARNING para silenciar las peticiones HTTP
logging.basicConfig(level=logging.WARNING, format='%(levelname)s - %(message)s')
"""

# 1. Conexión (usando tu método seguro)
connect_str = Path("key.txt").read_text(encoding="utf-8").strip()
blob_service_client = BlobServiceClient.from_connection_string(connect_str)
container_client = blob_service_client.get_container_client("zona-bronze")

directorio_origen = "../datos_locales"

# Iterar archivos locales con os.walk para recorrer directorios recursivamente
for root, dirs, files in os.walk(directorio_origen):
    for file in files:
        if file.endswith(".csv"):
            ruta_local = os.path.join(root, file)

            # Leer metadata del archivo (fecha de modificación) 
            timestamp_modificacion = os.path.getmtime(ruta_local)
            fecha = datetime.fromtimestamp(timestamp_modificacion)

            # Construir la ruta destino con formato "año/mes/día/archivo.csv"
            # Usamos :02d para asegurar que el mes 7 sea "07"
            adls_path = f"raw/ventas/{fecha.year}/{fecha.month:02d}/{fecha.day:02d}/{file}"

            # Lógica de reintentos por red inestable
            # definiremos cuántos intentos máximos tenemos hasta rendirnos 
            intentos_maximos = 3
            
            for intento in range(1, intentos_maximos + 1):
                try:
                    with open(ruta_local, "rb") as data: # apertura de archivo en modo lectura binaria
                        # Llamar a upload_blob con el nuevo path para subir cada archivo
                        container_client.upload_blob(name=adls_path, data=data, overwrite=True)
                    
                    # registro en consola que la operación fue un éxito
                    logging.info(f"Éxito: {file} -> {adls_path}")
                    break  # Salir del bucle de reintentos si funcionó
                    
                except Exception as e:
                    logging.warning(f"Fallo en red subiendo {file}. Intento {intento}/{intentos_maximos}. Error: {e}")
                    
                    if intento == intentos_maximos:
                        logging.error(f"ALERTA: Se abandonó la carga de {file} tras {intentos_maximos} intentos.")
                    else:
                        time.sleep(2)  # Pausa breve antes de intentar de nuevo

INFO - Request URL: 'https://datalakebsg2026.blob.core.windows.net/zona-bronze/raw/ventas/2023/10/15/ticket_0.csv'
Request method: 'PUT'
Request headers:
    'Content-Length': '21'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.30.0 Python/3.13.7 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': 'e4827b46-7b46-11f1-90cc-4438e85d1e74'
    'Authorization': 'REDACTED'
A body is sent with the request
INFO - Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Thu, 09 Jul 2026 03:33:04 GMT'
    'ETag': '"0x8DEDD6AC8E67704"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'be4e8c96-701e-0005-4a53-0f81ce000000'
    'x-ms-client-request-id': 'e4827b46-7b46-11f1-90cc-4438e85d1e74'
    'x-ms-version': 'REDACTED'
    'x-